# 목표

연말정산 신고 안내 문서 활용 RAG 시스템 구현

In [69]:
# 기본 경로 설정
# ===============================================================
import os
import requests

PROJECT_NUM = "14"

ROOT_DIR = os.getcwd()


try:
    from google.colab import drive, userdata
    IS_COLAB_MODE = True
    print("코랩 모드")

except ModuleNotFoundError as e:
    IS_COLAB_MODE = False
    ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))
    os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"로컬 모드")


DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(DATA_DIR, exist_ok=True)


# 환경변수 로드 설정
def get_secret(key_name: str):
    if IS_COLAB_MODE:
        return userdata.get(key_name)
    else:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=os.path.join(ROOT_DIR, ".env"))
        return os.getenv(key_name)


if IS_COLAB_MODE:
    import subprocess

    drive.mount('/content/drive')

    # 압축 파일 확보
    if "raw.tar.gz" not in os.listdir():

        headers = {
            "Authorization": f"token {get_secret('GITHUB_PAT')}",
            "Accept": "application/vnd.github.v3+json"
        }

        release_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/tags/data-v{PROJECT_NUM}"

        response = requests.get(release_url, headers=headers)
        asset_id = response.json().get('assets', [])[0]["id"]

        download_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/assets/{asset_id}"
        download_headers = headers.copy()
        download_headers["Accept"] = "application/octet-stream"

        with requests.get(download_url, headers=download_headers, stream=True) as r:
            r.raise_for_status()
            with open("raw.tar.gz", 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("· 압축 파일 있음")


    # 압축 해제 및 경로 처리
    if not os.path.exists(RAW_DIR):
        subprocess.run(["tar", "-xzvf", "raw.tar.gz", "-C", DATA_DIR], check=True)

    print("· 압축 해제 완료")
    print("· 환경 세팅 완료")


    DRIVE_DIR = os.path.join(ROOT_DIR, "drive", "MyDrive")
    SAVE_DIR = os.path.join(DRIVE_DIR, "runs", PROJECT_NUM)

    os.makedirs(SAVE_DIR, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "2024년+원천징수의무자를+위한+연말정산+신고안내.pdf")

로컬 모드


### 텍스트 데이터

In [70]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
TEXTS_BY_PAGE = loader.load()

In [71]:
for i, page in enumerate(TEXTS_BY_PAGE):
    page.metadata = {"page": i}

### 표 데이터

In [99]:
from img2table.document import PDF
from img2table.ocr import TesseractOCR

pdf = PDF(
    PDF_PATH, 
    detect_rotation=False,
    pdf_text_extraction=True
)

ocr = TesseractOCR(n_threads=1, lang="eng")

TABLES_BY_PAGE = pdf.extract_tables(
    ocr=ocr,
    implicit_rows=False,
    implicit_columns=False,
    borderless_tables=False,
    min_confidence=50
)

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.54 : libtiff 4.7.1 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.5 zlib/1.2.12 liblzma/5.8.2 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.1 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.67.1


In [104]:
# 표 데이터 - 메타데이터 title 보정

TABLE_TITLE_LIST = list()
for page_num, tables in TABLES_BY_PAGE.items():
    for table in tables:
        if table and table.title:
            if len(table.title) > 20:
                table.title = None
            else:
                TABLE_TITLE_LIST.append(table.title)

In [ ]:
# 표 데이터 metadata에 merge
import pandas as pd

def trim_table(df: pd.DataFrame):
    df.columns = df.iloc[0]
    df = df[1:]
    df.reset_index(drop=True, inplace=True)

    return df


to_delete_list = list()

for page_num, tables in TABLES_BY_PAGE.items():
    if tables:
        TEXTS_BY_PAGE[page_num].metadata["table"] = {
            f"{page_num}.{i}": trim_table(table.df) for i, table in enumerate(tables)
        }

        for table in tables:
            table = trim_table(table.df)

    else:
        TEXTS_BY_PAGE[page_num].metadata["table"] = None
        to_delete_list.append(page_num)

for page_num in to_delete_list:
    del TABLES_BY_PAGE[page_num]

In [108]:
TABLES_BY_PAGE[7][1].df

,0,1,2,3,4,5
0,구 분,구 분,기본공제대상자의 요건,기본공제대상자의 요건,근로기간 지출한\n비용만 공제,비 고
1,구 분,구 분,나이요건 소득요건,나이요건 소득요건,근로기간 지출한\n비용만 공제,비 고
2,특별 소득공제,보 험 료,None,None,None,None
3,특별 소득공제,주택자금공제,None,None,None,None
4,그 밖의 소득공제,개인연금저축,None,None,None,None
5,그 밖의 소득공제,주택마련저축,None,None,None,None
6,그 밖의 소득공제,신용카드 등,×,,,None
7,자녀세액공제 (8세이상),자녀세액공제 (8세이상),,,-,기본공제대상 자녀\n(입양자·위탁아동·손자녀 포함)


pdf 육안 확인 + df로 바꿔보니 여간 복잡한 게 아니다.

1. 병합된 셀이 많다.
2. 특수문자(O) 같은 게 씹히는 경우가 있다.
3. 타이틀을 일일이 달아줘야 할 것 같다.
4. 표에서 확인할 수 있는 정보는 표를 참고하라고 따로 명령해야 할 듯.

큰 제목

내용 1

내용 2

    - 하위 내용 1

    - 하위 내용 2

      - 표 1

...

In [103]:
"https://huggingface.co/microsoft/tapex-base-finetuned-wikisql"

'https://huggingface.co/microsoft/tapex-base-finetuned-wikisql'

## RAG 설계

0. 문장 데이터와 표 데이터로 나눈다.
1. 문장 데이터는 OPENAI 모델이 진행.
2. 표 데이터는 https://huggingface.co/microsoft/tapex-base-finetuned-wikisql
3. 허위 정보를 알려줘서는 안 되니, 모르는 건 모른다고 하자.
4. 참고한 페이지 정보를 같이 출력해주어 유저가 더블 체크할 수 있도록 하자.